<a href="https://colab.research.google.com/github/rahmatnug/capstone-cangkringan-ml/blob/main/training_model_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("=== SPRINT 2: Training Model Time-Series ===")
print("Guardrail: Menggunakan TimeSeriesSplit (k=5) - NO RANDOM SPLIT\n")

# 1. Load Data dari hasil Feature Engineering (Notebook 2)
df = pd.read_csv('features_demand_data.csv')

# Sort berdasarkan tanggal untuk memastikan urutan waktu benar (Wajib untuk TimeSeriesSplit)
df['tanggal_permintaan'] = pd.to_datetime(df['tanggal_permintaan'])
df = df.sort_values(by=['tanggal_permintaan', 'id_poktan', 'id_komoditas']).reset_index(drop=True)

# 2. Preprocessing (Encoding & Drop Kolom yang tidak dipakai training)
# One-Hot Encoding untuk fitur musim (Rendeng, Gadu, Bera)
df_encoded = pd.get_dummies(df, columns=['fase_musim'], drop_first=False)

# Tentukan Target (y) dan Fitur (X)
y = df_encoded['volume_permintaan']
# Buang ID transaksi, tanggal, status, dan target dari fitur
X = df_encoded.drop(columns=['id_permintaan', 'tanggal_permintaan', 'status_data', 'volume_permintaan'])

# 3. Setup Time-Series Cross Validation (k=5)
tscv = TimeSeriesSplit(n_splits=5)

# Inisialisasi Model (Tuned Hyperparameters untuk redam noise & tangkap tren)
models = {
    "1. Baseline (Dummy Mean)": DummyRegressor(strategy="mean"),
    "2. Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=4, random_state=42, n_jobs=-1),
    "3. XGBoost Regressor": xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='reg:squarederror'
    )
}

# 4. Training & Evaluasi Loop
results = []

for name, model in models.items():
    fold_r2, fold_mape, fold_mae, fold_rmse = [], [], [], []

    # Loop melalui 5 lipatan (folds) waktu
    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # Training
        model.fit(X_train, y_train)

        # Inferensi
        y_pred = model.predict(X_test)

        # Menghindari prediksi negatif
        y_pred = np.maximum(0, y_pred)

        # Evaluasi Metrik
        fold_r2.append(r2_score(y_test, y_pred))
        fold_mape.append(mean_absolute_percentage_error(y_test, y_pred))
        fold_mae.append(mean_absolute_error(y_test, y_pred))
        fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))

    # Simpan rata-rata metrik dari 5 folds
    results.append({
        "Model": name,
        "R2 Score": np.mean(fold_r2),
        "MAPE (%)": np.mean(fold_mape) * 100,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse)
    })

# 5. Tampilkan Hasil Akhir Evaluasi Global
df_results = pd.DataFrame(results)
print("=== Hasil Evaluasi Metrik Lintas 5 Folds ===")
print(df_results.to_string(index=False))

print("\nCek Status Threshold SRS:")
xgboost_r2 = df_results[df_results['Model'] == '3. XGBoost Regressor']['R2 Score'].values[0]
xgboost_mape = df_results[df_results['Model'] == '3. XGBoost Regressor']['MAPE (%)'].values[0]

if xgboost_r2 >= 0.70 and xgboost_mape <= 20.0:
    print(f"✅ XGBoost LOLOS Kriteria Uji (R2: {xgboost_r2:.2f}, MAPE: {xgboost_mape:.2f}%)\n")
else:
    print(f"⚠️ XGBoost BELUM LOLOS (R2: {xgboost_r2:.2f}, MAPE: {xgboost_mape:.2f}%). Perlu tuning.\n")


# 6. EKSPOR ARTEFAK FINAL
print("=== Mengekspor Artefak Final ===")
# Ambil model XGBoost langsung dari memori (tanpa load)
model_xgb = models["3. XGBoost Regressor"]

# Ekspor Model
joblib.dump(model_xgb, 'model_demand.pkl')

# Ekspor Pipeline Fitur
pipeline_artefak = {
    'fitur_input': list(X.columns),
    'keterangan': 'Wahyu, gunakan list fitur_input ini untuk df.reindex(columns=fitur_input, fill_value=0) di Streamlit.'
}
joblib.dump(pipeline_artefak, 'pipeline.pkl')
print("✅ Artefak 'model_demand.pkl' & 'pipeline.pkl' berhasil diekspor!\n")


# 7. ANALISIS ERROR PER SKU (PRODUK)
print("=== Analisis Error Per Produk (SKU) ===")
# Lakukan Prediksi dan simpan hasilnya
df_encoded['prediksi_volume'] = np.maximum(0, model_xgb.predict(X))

skus = {
    1: 'GB Propunic', 2: 'GB Profeed', 3: 'GB Proquatic',
    4: 'Pendawa Subur POC', 5: 'Compossap', 6: 'Agen Hayati'
}

hasil_analisis = []
for sku_id, sku_name in skus.items():
    mask = df_encoded['id_komoditas'] == sku_id
    y_true = df_encoded[mask]['volume_permintaan']
    y_pred = df_encoded[mask]['prediksi_volume']

    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    hasil_analisis.append({
        'ID': sku_id,
        'Nama Produk': sku_name,
        'R2 Score': round(r2, 4),
        'MAPE (%)': round(mape, 2),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2)
    })

df_analisis = pd.DataFrame(hasil_analisis)
print(df_analisis.to_string(index=False))

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

print("=== SPRINT 2: Analisis Error Per Produk (SKU) ===")

# 1. Load Data dan Model yang udah diekspor tadi
df = pd.read_csv('features_demand_data.csv')
df['tanggal_permintaan'] = pd.to_datetime(df['tanggal_permintaan'])
df = df.sort_values(by=['tanggal_permintaan', 'id_poktan', 'id_komoditas']).reset_index(drop=True)

# Encoding fitur yang sama seperti saat training
df_encoded = pd.get_dummies(df, columns=['fase_musim'], drop_first=False)
X = df_encoded.drop(columns=['id_permintaan', 'tanggal_permintaan', 'status_data', 'volume_permintaan'])

model_xgb = joblib.load('xgboost_demand_model.pkl')

# 2. Lakukan Prediksi
df['prediksi_volume'] = np.maximum(0, model_xgb.predict(X))

# 3. Mapping ID ke Nama Produk untuk Laporan
skus = {
    1: 'GB Propunic', 2: 'GB Profeed', 3: 'GB Proquatic',
    4: 'Pendawa Subur POC', 5: 'Compossap', 6: 'Agen Hayati'
}

# 4. Hitung Metrik per Produk
hasil_analisis = []
for sku_id, sku_name in skus.items():
    mask = df['id_komoditas'] == sku_id
    y_true = df[mask]['volume_permintaan']
    y_pred = df[mask]['prediksi_volume']

    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    hasil_analisis.append({
        'ID': sku_id,
        'Nama Produk': sku_name,
        'R2 Score': round(r2, 4),
        'MAPE (%)': round(mape, 2),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2)
    })

df_analisis = pd.DataFrame(hasil_analisis)
print(df_analisis.to_string(index=False))

=== SPRINT 2: Analisis Error Per Produk (SKU) ===


FileNotFoundError: [Errno 2] No such file or directory: 'xgboost_demand_model.pkl'

In [ ]:
import joblib

# 1. Ekspor (Rename) Model sesuai permintaan Task
joblib.dump(model, 'model_demand.pkl')

# 2. Buat "Pipeline" Artefak untuk Streamlit
# Menyimpan daftar urutan kolom yang wajib ada saat inferensi
expected_columns = list(X.columns)

pipeline_artefak = {
    'fitur_input': expected_columns,
    'keterangan': 'Wahyu, gunakan list fitur_input ini untuk df.reindex(columns=fitur_input, fill_value=0) di Streamlit biar shape XGBoost gak error saat get_dummies.'
}

joblib.dump(pipeline_artefak, 'pipeline.pkl')

print("✅ Artefak Final berhasil diekspor: 'model_demand.pkl' dan 'pipeline.pkl'")